# Agent memory on Databricks, step by step

A model doesn't remember anything between requests. This notebook builds memory for an agent one layer at a time,
using **Databricks Agent Memory (Beta)**. Every store here lives in **Lakebase**, serverless Postgres in your workspace.

| Kind of memory | What it holds | Where it lives here |
|---|---|---|
| Working memory | everything the model sees on one call | the context window |
| Episodic memory | what happened in one conversation, step by step | a **session store** |
| Semantic memory | facts: who the user is, what a metric means | a **memory store** |
| Procedural memory | how to do a task: a query recipe that worked | a **memory store** |

The example is **Bean & Byte**, a New York coffee chain. Maya manages one of its stores and asks an ops assistant
about her sales. Her team counts "weekly sales" in a way nothing in the database says.

**You need:** a workspace with Agent Memory (Beta) enabled, a Databricks CLI profile, and access to a `system.ai`
chat model through the AI Gateway. Run it locally (Python 3.11+) or in a Databricks notebook.

In [1]:
# %pip install -r ../requirements.txt     # uncomment on first run, then restart the kernel
import json, os, pathlib, subprocess, sys, time, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "DEFAULT")   # your CLI profile; not needed inside Databricks
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

PREFIX = "bean-byte-nb"      # every store this notebook creates starts with this
USER = "maya"                # the actor: whose memory we read and write

from databricks_agentkit import AgentKitClient, workspace_client
ak = AgentKitClient(workspace_client())
print("workspace:", workspace_client().config.host)

workspace: https://<your-workspace>.cloud.databricks.com


## 1. The model forgets

Two calls to the same model. The second call has no idea what the first one said.

In [2]:
from databricks_langchain import ChatDatabricks
from databricks_agentkit.langgraph.messages import message_items

llm = ChatDatabricks(endpoint="system.ai.claude-sonnet-5-5", use_ai_gateway=True)

def text(reply):
    content = message_items(reply)[-1]["content"]
    if isinstance(content, list):
        content = "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
    return content.strip()

first = llm.invoke("Hi, I'm Maya. I manage the Park Slope store. Reply in one sentence.")
second = llm.invoke("Which store do I manage? Reply in one sentence.")
print("call 1:", text(first))
print("call 2:", text(second))

call 1: Hi Maya, nice to meet you! What can I help you with for the Park Slope store today?
call 2: I don't know which store you manage, since nothing in our conversation mentions it. If you tell me the store's name or location, I can help from there.


Chat apps look like they remember because the **app resends the whole conversation** on every call.
That is working memory: it works, but it grows with every turn, and it's gone when the process ends.

In [3]:
history = [("user", "Hi, I'm Maya. I manage the Park Slope store. Reply in one sentence."),
           ("assistant", text(first)),
           ("user", "Which store do I manage? Reply in one sentence.")]
third = llm.invoke(history)
print("with history:", text(third))
print("input tokens: call 2 =", second.usage_metadata["input_tokens"], "· with history =", third.usage_metadata["input_tokens"])

with history: You mentioned you manage the Park Slope store.
input tokens: call 2 = 25 · with history = 90


## 2. A session store: one conversation, saved in Lakebase

A session store keeps the ordered history of a conversation: messages, tool calls and results.
It is episodic memory. Any process can pick the conversation up again.

In [4]:
sessions = ak.session_stores.create(f"{PREFIX}-sessions")
chat = sessions.add(actor_id=USER, session_id="nb-chat-1")
chat.append_items([
    {"type": "message", "role": "user", "content": "Hi, I'm Maya, I manage the Park Slope store."},
    {"type": "message", "role": "assistant", "content": "Hi Maya! What would you like to know about Park Slope?"},
])
for item in sessions.get("nb-chat-1").list_items():      # read it back, as a new process would
    print(item.data["role"], "›", item.data["content"])

assistant › Hi Maya! What would you like to know about Park Slope?
user › Hi, I'm Maya, I manage the Park Slope store.


A session can be **forked**: a new, independent conversation that starts from the same history.
Useful for trying a different answer without touching the original.

In [5]:
fork = chat.fork(actor_id=USER, session_id="nb-chat-1-fork")
fork.append_items([{"type": "message", "role": "user", "content": "Actually, show me SoHo instead."}])
print("original:", len(list(chat.list_items())), "items · fork:", len(list(fork.list_items())), "items")

original: 2 items · fork: 3 items


## 3. A memory store: facts that outlive any one conversation

A memory store holds durable facts per actor, as small files under paths like `/profile.md`.
Search is hybrid: keywords **and** meaning, so a question can find a fact that uses different words.
The ranking blends both: a result that matches on keywords and on meaning ranks highest.

In [6]:
memory = ak.memory_stores.create(f"{PREFIX}-memory")
memory.put(actor_id=USER, path="/profile.md", content="Maya manages the Park Slope store (loc_code BK-PS).")
memory.put(actor_id=USER, path="/definitions.md",
           content="Weekly sales = sale transactions (txn_type 'S') from the counter and the app only. "
                   "Gift cards, refunds and delivery are reported separately.")
memory.put(actor_id=USER, path="/preferences.md", content="Keep answers short: one small table.")
memory.put(actor_id="devon", path="/profile.md", content="Devon manages the DUMBO store.")

for query in ["how do we count revenue",     # no words in common with the fact: found by meaning
              "where does Maya work"]:       # "Maya" also matches as a keyword
    print(f"search {query!r}:")
    top = memory.search(actor_id=USER, query=query, limit=1)[0]
    print(f"   top hit: {top.memory.path:16} {top.memory.content[:56]}")

search 'how do we count revenue':


   top hit: /definitions.md  Weekly sales = sale transactions (txn_type 'S') from the
search 'where does Maya work':


   top hit: /profile.md      Maya manages the Park Slope store (loc_code BK-PS).


Memory is partitioned by **actor**. Devon's search never sees Maya's facts. The actor must come from a trusted
source (the signed-in user), never from the model.

In [7]:
print([m.path + ": " + m.content for m in memory.list(actor_id="devon")])
print([r.memory.path for r in memory.search(actor_id="devon", query="weekly sales definition", limit=5)])

['/profile.md: Devon manages the DUMBO store.']


['/profile.md']


## 4. An agent that uses both

The agent in `agent/agent.py` is a plain LangGraph agent with three database tools. Two lines give it memory:
`checkpointer()` saves each step to the session store, and `memory_tools(actor)` adds `remember`, `recall`,
`forget` and `list_memories`. Both read their store from an environment variable.

Here the agent only **reads** memory (`AGENT_MEMORY_WRITES=pipeline`), so every run of this notebook is repeatable.
The terminal walkthrough in the README shows it writing memory with `remember` as well.

In [8]:
os.environ["AGENT_SESSION_STORE"] = sessions.name
os.environ["AGENT_MEMORY_STORE"] = memory.id
os.environ["AGENT_MEMORY_WRITES"] = "pipeline"   # read-only: recall and list_memories
from langchain_core.messages import AIMessage
from databricks_agentkit.langgraph import thread_config
from agent.agent import build_agent
from agent.memory import memory_index

def ask(session_id, question, user=USER):
    agent = build_agent(user, memory_index(user))          # a fresh agent each time, like a restart
    t0, calls, tokens = time.time(), [], 0
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, thread_config(session_id, user))
    new = result["messages"][[i for i, m in enumerate(result["messages"]) if m.type == "human"][-1]:]
    for m in new:
        if isinstance(m, AIMessage):
            tokens += (m.usage_metadata or {}).get("input_tokens", 0)
            calls += [f"{c['name']}" for c in m.tool_calls]
    print("tool calls:", " → ".join(calls) or "none")
    print(text(new[-1]))                                     # the final answer
    print(f"[{len(calls)} tool calls · {tokens:,} tokens in · {time.time() - t0:.1f} s]")

ask("nb-chat-2", "How did my store do last week?")

tool calls: recall → list_tables → describe_table → describe_table → run_sql
Park Slope (BK-PS), Nov 2–8:

| Channel | Sales txns | Sales (USD) |
|---|---|---|
| Counter | 1,118 | $7,530.10 |
| App | 565 | $3,681.55 |
| **Total** | **1,683** | **$11,211.65** |

Following your team's definition, this counts sales from the counter and app only. Gift cards, refunds and delivery are left out.
[5 tool calls · 8,952 tokens in · 12.3 s]


It recalled Maya's store, her team's definition and her preferred format, without asking.
Now a follow-up in the **same conversation**, from a brand-new agent object: the history comes back from the
session store.

In [9]:
ask("nb-chat-2", "And the week before that?")

tool calls: run_sql
Park Slope (BK-PS), Oct 26–Nov 1:

| Channel | Sales txns | Sales (USD) |
|---|---|---|
| Counter | 1,106 | $7,435.00 |
| App | 498 | $3,297.30 |
| **Total** | **1,604** | **$10,732.30** |

Last week was up $479.35 (+4.5%) and 79 transactions. Most of the gain came from the app, which rose $384.25.
[1 tool calls · 5,275 tokens in · 6.4 s]


## 5. A memory pipeline: learning after the conversation

Writing memory during a chat costs tool calls, and the model has to notice what's worth keeping.
A **memory pipeline** (the "Dreamer") does it afterwards: it reads plain transcripts from a session store,
extracts durable facts, merges duplicates and replaces facts that changed. A short instructions file
(`dreamer.md`) tells it which files to keep and how to write them.

Pipelines skip tool calls and LangGraph checkpoints, so we give them a separate **transcript store** with just
the user and assistant messages. They also only read turns older than 60 seconds.

In [10]:
transcripts = ak.session_stores.create(f"{PREFIX}-transcripts")
t = transcripts.add(actor_id=USER, session_id="nb-chat-3")
t.append_items([
    {"type": "message", "role": "user",
     "content": "Quick heads up: as of this week I run the Williamsburg store. Devon took over Park Slope."},
    {"type": "message", "role": "assistant", "content": "Thanks, Maya. I'll use Williamsburg (BK-WB) from now on."},
])
print("before:", next(memory.list(actor_id=USER, path="/profile.md")).content)
time.sleep(65)   # the pipeline only reads turns older than 60 s

before: Maya manages the Park Slope store (loc_code BK-PS).


In [11]:
def agentbricks(*args):
    profile = ["-p", os.environ["DATABRICKS_CONFIG_PROFILE"]] if os.environ.get("DATABRICKS_CONFIG_PROFILE") else []
    out = subprocess.run(["agentbricks", *profile, "-o", "json", *args], capture_output=True, text=True, check=True,
                         env=dict(os.environ, AGENTBRICKS_DISABLE_TELEMETRY="1", AGENTBRICKS_DISABLE_AGENT_HINT="1"))
    return json.loads(out.stdout) if out.stdout.strip() else {}

print(open(REPO / "dreamer.md").read())                     # instructions that steer what it keeps
pipeline = agentbricks("memory", "pipeline", "create", "--memory-store", memory.id,
                       "--session-store", transcripts.name, "--display-name", f"{PREFIX}-dreamer",
                       "--instructions", f"@{REPO / 'dreamer.md'}")["name"]
run = agentbricks("memory", "pipeline", "run", pipeline)["name"]
t0 = time.time()
while (state := agentbricks("memory", "pipeline", "runs", "get", run).get("state")) in ("RUNNING", "PENDING", None):
    time.sleep(10)
print(f"{state} after {time.time() - t0:.0f} s")

These are chats between a Bean & Byte store manager and the ops assistant.
Keep only durable facts the assistant will need in future chats:
- /profile.md: who the user is and which store they manage,
  written about the user ("The user is Maya ...").
- /definitions.md: how their team defines metrics.
- /preferences.md: how they like answers.
- /workflows.md: query recipes that worked.
When a fact changes, replace it and note what it replaced.
Never keep sales numbers; they change every week.



SUCCEEDED after 106 s


In [12]:
for m in memory.list(actor_id=USER):
    print(f"{m.path:16} {m.content}")

/profile.md      Maya manages the Williamsburg store (store code BK-WB) as of this week. She previously managed the Park Slope store (loc_code BK-PS), but has since moved to Williamsburg.

Devon took over management of the Park Slope store from Maya.
/definitions.md  Weekly sales = sale transactions (txn_type 'S') from the counter and the app only. Gift cards, refunds and delivery are reported separately.
/preferences.md  Keep answers short: one small table.


The pipeline rewrote Maya's profile: Williamsburg now, Park Slope before. The next conversation uses it.

In [13]:
ask("nb-chat-4", "How did my store do last week?")

tool calls: recall → list_tables → describe_table → describe_table → run_sql → run_sql
Williamsburg (BK-WB), last week (Nov 2–8):

| Channel | Transactions | Sales |
|---|---|---|
| Counter | 1,366 | $9,056.00 |
| App | 664 | $4,403.50 |
| **Total** | **2,030** | **$13,459.50** |

Delivery is left out because your team reports it separately. It came to 200 transactions and $1,381.10.
[6 tool calls · 11,595 tokens in · 11.6 s]


## 6. Under the hood: Lakebase

Each store is its own Postgres database on a Lakebase branch. The store tells you where.
(The tables are an internal detail of the managed service: read them to learn, build on the SDK.)

In [14]:
print(ak.memory_stores.get(memory.id).storage_backend)
os.environ["AGENT_MEMORY_STORE"] = memory.id
import lakebase
from rich.console import Console
lakebase.con = Console(color_system=None)   # plain text in the notebook
with lakebase.connect(lakebase.memory_db()) as c:
    for row in c.execute("SELECT path, content_vector IS NOT NULL AS embedding, content_tsvector IS NOT NULL AS keywords "
                         "FROM memory.memory_entries WHERE actor_id = %s ORDER BY path", (USER,)):
        print(row)

{'backend_type': 'STORAGE_BACKEND_TYPE_LAKEBASE', 'backend_id': 'projects/<lakebase-project>/branches/production/databases/memory-bean-byte-nb-memory'}


Lakebase project  <lakebase-project>  branch production

database          memory-bean-byte-nb-memory

('/definitions.md', True, True)
('/preferences.md', True, True)
('/profile.md', True, True)


## 7. Clean up

Deleting a session store does **not** delete memories distilled from it, so delete both.

In [15]:
agentbricks("memory", "pipeline", "delete", pipeline, "--yes")
memory.delete()
for store in (sessions, transcripts):
    for s in store.list():
        s.delete(force=True)
    store.delete()
print("deleted", f"{PREFIX}-*")

deleted bean-byte-nb-*
